# Validate Final Event Metadata

## Objective
This notebook validates the canonical event metadata file:

```text
data/processed/event_metadata_final.csv
```

The goals are to verify that:

1. all transcript timestamps were parsed successfully
2. GMT timestamps were converted correctly to `America/New_York`
3. the `after_market_close_et` flag is correct
4. the final event day assignment is consistent with the rule:
   - if the call occurred strictly after 4:00 PM ET, assign the next business day
   - otherwise assign the same calendar day
5. the final event day is a business day

This notebook is for **audit and reporting only**. It does not modify the pipeline output.

---

In [1]:
import pandas as pd
from pathlib import Path
EVENT_METADATA_PATH = Path("../data/processed/event_metadata_final.csv")

df = pd.read_csv(EVENT_METADATA_PATH)
df.head()

,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error
0,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,22:00:00,17:00:00,True,2016-01-27,NaN
1,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-04-27,NaN
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-07-27,NaN
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,21:00:00,17:00:00,True,2016-10-26,NaN
4,AAPL,2017-Jan-31-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,2017,Q1,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,22:00:00,17:00:00,True,2017-02-01,NaN


## 1. Basic structure checks

In [2]:
df.shape

(188, 15)

In [3]:
df.columns.tolist()

['ticker',
 'file_name',
 'file_path',
 'call_date_from_filename',
 'year',
 'quarter_label',
 'call_datetime_header_raw',
 'call_datetime_parsed',
 'call_datetime_gmt',
 'call_datetime_et',
 'call_time_gmt',
 'call_time_et',
 'after_market_close_et',
 'event_trading_day_final',
 'error']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ticker                    188 non-null    str    
 1   file_name                 188 non-null    str    
 2   file_path                 188 non-null    str    
 3   call_date_from_filename   188 non-null    str    
 4   year                      188 non-null    int64  
 5   quarter_label             188 non-null    str    
 6   call_datetime_header_raw  188 non-null    str    
 7   call_datetime_parsed      188 non-null    str    
 8   call_datetime_gmt         188 non-null    str    
 9   call_datetime_et          188 non-null    str    
 10  call_time_gmt             188 non-null    str    
 11  call_time_et              188 non-null    str    
 12  after_market_close_et     188 non-null    bool   
 13  event_trading_day_final   188 non-null    str    
 14  error                

## 2. Check for parsing errors

In [5]:
df["error"].isna().value_counts(dropna=False)
error_rows = df[df["error"].notna()].copy()
error_rows.head(20)

,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error


## 3. Parse timezone-aware timestamps from the saved metadata

In [6]:
df["call_datetime_gmt_ts"] = pd.to_datetime(df["call_datetime_gmt"], errors="coerce", utc=True)
df["call_datetime_et_ts"] = pd.to_datetime(df["call_datetime_et"], errors="coerce", utc=True)

# Convert ET timestamps back to America/New_York for validation display
df["call_datetime_et_ts"] = df["call_datetime_gmt_ts"].dt.tz_convert("America/New_York")

In [7]:
df[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_gmt",
    "call_datetime_et",
    "call_datetime_gmt_ts",
    "call_datetime_et_ts"
]].head(10)

,ticker,file_name,call_datetime_header_raw,call_datetime_gmt,call_datetime_et,call_datetime_gmt_ts,call_datetime_et_ts
0,AAPL,2016-Jan-26-AAPL.txt,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,2016-01-26 22:00:00+00:00,2016-01-26 17:00:00-05:00
1,AAPL,2016-Apr-26-AAPL.txt,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,2016-04-26 21:00:00+00:00,2016-04-26 17:00:00-04:00
2,AAPL,2016-Jul-26-AAPL.txt,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,2016-07-26 21:00:00+00:00,2016-07-26 17:00:00-04:00
3,AAPL,2016-Oct-25-AAPL.txt,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,2016-10-25 21:00:00+00:00,2016-10-25 17:00:00-04:00
4,AAPL,2017-Jan-31-AAPL.txt,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,2017-01-31 22:00:00+00:00,2017-01-31 17:00:00-05:00
5,AAPL,2017-May-02-AAPL.txt,"MAY 02, 2017 / 9:00PM GMT",2017-05-02T21:00:00+00:00,2017-05-02T17:00:00-04:00,2017-05-02 21:00:00+00:00,2017-05-02 17:00:00-04:00
6,AAPL,2017-Aug-01-AAPL.txt,"AUGUST 01, 2017 / 9:00PM GMT",2017-08-01T21:00:00+00:00,2017-08-01T17:00:00-04:00,2017-08-01 21:00:00+00:00,2017-08-01 17:00:00-04:00
7,AAPL,2017-Nov-02-AAPL.txt,"NOVEMBER 02, 2017 / 9:00PM GMT",2017-11-02T21:00:00+00:00,2017-11-02T17:00:00-04:00,2017-11-02 21:00:00+00:00,2017-11-02 17:00:00-04:00
8,AAPL,2018-Feb-01-AAPL.txt,"FEBRUARY 01, 2018 / 10:00PM GMT",2018-02-01T22:00:00+00:00,2018-02-01T17:00:00-05:00,2018-02-01 22:00:00+00:00,2018-02-01 17:00:00-05:00
9,AAPL,2018-May-01-AAPL.txt,"MAY 01, 2018 / 9:00PM GMT",2018-05-01T21:00:00+00:00,2018-05-01T17:00:00-04:00,2018-05-01 21:00:00+00:00,2018-05-01 17:00:00-04:00


## 4. Recompute after-market-close flag independently

In [8]:
market_close_ts = df["call_datetime_et_ts"].dt.normalize() + pd.Timedelta(hours=16)

df["after_market_close_et_recomputed"] = df["call_datetime_et_ts"] > market_close_ts

In [9]:
df[[
    "ticker",
    "file_name",
    "call_datetime_et_ts",
    "after_market_close_et",
    "after_market_close_et_recomputed"
]].head(20)

,ticker,file_name,call_datetime_et_ts,after_market_close_et,after_market_close_et_recomputed
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-26 17:00:00-05:00,True,True
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-26 17:00:00-04:00,True,True
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-26 17:00:00-04:00,True,True
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-25 17:00:00-04:00,True,True
4,AAPL,2017-Jan-31-AAPL.txt,2017-01-31 17:00:00-05:00,True,True
5,AAPL,2017-May-02-AAPL.txt,2017-05-02 17:00:00-04:00,True,True
6,AAPL,2017-Aug-01-AAPL.txt,2017-08-01 17:00:00-04:00,True,True
7,AAPL,2017-Nov-02-AAPL.txt,2017-11-02 17:00:00-04:00,True,True
8,AAPL,2018-Feb-01-AAPL.txt,2018-02-01 17:00:00-05:00,True,True
9,AAPL,2018-May-01-AAPL.txt,2018-05-01 17:00:00-04:00,True,True


In [10]:
df["after_market_close_matches"] = (
    df["after_market_close_et"].astype("boolean") ==
    df["after_market_close_et_recomputed"].astype("boolean")
)

df["after_market_close_matches"].value_counts(dropna=False)

after_market_close_matches
True    188
Name: count, dtype: Int64

## 5. Recompute expected final event day independently

This check reproduces the pipeline logic:
- after 4:00 PM ET → next business day
- otherwise → same ET calendar day

Note: `BDay` handles weekends, not exchange holidays.

In [11]:
from pandas.tseries.offsets import BDay

In [12]:
def recompute_expected_event_day(call_dt_et):
    if pd.isna(call_dt_et):
        return pd.NaT

    market_close = call_dt_et.normalize() + pd.Timedelta(hours=16)

    if call_dt_et > market_close:
        return (call_dt_et.normalize() + BDay(1)).date()
    return call_dt_et.date()

df["expected_event_trading_day"] = df["call_datetime_et_ts"].apply(recompute_expected_event_day)
df["expected_event_trading_day"] = pd.to_datetime(df["expected_event_trading_day"]).dt.date.astype(str)

In [13]:
df[[
    "ticker",
    "file_name",
    "call_datetime_et_ts",
    "after_market_close_et",
    "event_trading_day_final",
    "expected_event_trading_day"
]].head(20)

,ticker,file_name,call_datetime_et_ts,after_market_close_et,event_trading_day_final,expected_event_trading_day
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-26 17:00:00-05:00,True,2016-01-27,2016-01-27
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-26 17:00:00-04:00,True,2016-04-27,2016-04-27
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-26 17:00:00-04:00,True,2016-07-27,2016-07-27
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-25 17:00:00-04:00,True,2016-10-26,2016-10-26
4,AAPL,2017-Jan-31-AAPL.txt,2017-01-31 17:00:00-05:00,True,2017-02-01,2017-02-01
5,AAPL,2017-May-02-AAPL.txt,2017-05-02 17:00:00-04:00,True,2017-05-03,2017-05-03
6,AAPL,2017-Aug-01-AAPL.txt,2017-08-01 17:00:00-04:00,True,2017-08-02,2017-08-02
7,AAPL,2017-Nov-02-AAPL.txt,2017-11-02 17:00:00-04:00,True,2017-11-03,2017-11-03
8,AAPL,2018-Feb-01-AAPL.txt,2018-02-01 17:00:00-05:00,True,2018-02-02,2018-02-02
9,AAPL,2018-May-01-AAPL.txt,2018-05-01 17:00:00-04:00,True,2018-05-02,2018-05-02


## 6. Compare pipeline output vs independently recomputed event day

In [14]:
df["event_day_matches"] = df["event_trading_day_final"] == df["expected_event_trading_day"]
df["event_day_matches"].value_counts(dropna=False)

event_day_matches
True    188
Name: count, dtype: int64

In [15]:
(df["event_day_matches"].value_counts(normalize=True, dropna=False) * 100).round(2)

event_day_matches
True    100.0
Name: proportion, dtype: float64

In [16]:
mismatches = df.loc[~df["event_day_matches"]].copy()

mismatches[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_et_ts",
    "after_market_close_et",
    "event_trading_day_final",
    "expected_event_trading_day",
    "error"
]].sort_values(["ticker", "file_name"]).head(50)

,ticker,file_name,call_datetime_header_raw,call_datetime_et_ts,after_market_close_et,event_trading_day_final,expected_event_trading_day,error


## 7. Validate that final event days are business days

In [17]:
min_date = pd.to_datetime(df["event_trading_day_final"], errors="coerce").min() - pd.Timedelta(days=10)
max_date = pd.to_datetime(df["event_trading_day_final"], errors="coerce").max() + pd.Timedelta(days=10)

business_days = pd.bdate_range(start=min_date, end=max_date)
business_days_set = set(business_days.date)

df["event_trading_day_final_dt"] = pd.to_datetime(df["event_trading_day_final"], errors="coerce")
df["event_day_is_business_day"] = df["event_trading_day_final_dt"].dt.date.isin(business_days_set)

df["event_day_is_business_day"].value_counts(dropna=False)

event_day_is_business_day
True    188
Name: count, dtype: int64

In [18]:
non_business_days = df.loc[~df["event_day_is_business_day"]].copy()

non_business_days[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "event_trading_day_final"
]].head(50)

,ticker,file_name,call_datetime_header_raw,event_trading_day_final


## 8. Focus check on edge-case times

These times are especially important because DST often changes whether they fall before or after market close.

In [20]:
df["gmt_hour_minute"] = df["call_datetime_gmt_ts"].dt.strftime("%H:%M")

edge_cases = df[df["gmt_hour_minute"].isin(["20:30", "21:00", "21:30", "22:00", "22:30"])]

edge_cases_sorted = edge_cases.sort_values(["gmt_hour_minute", "ticker"])

edge_cases_sorted[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_gmt_ts",
    "call_datetime_et_ts",
    "gmt_hour_minute",
    "after_market_close_et",
    "event_trading_day_final",
    "expected_event_trading_day",
    "event_day_matches"
]].head(100)

,ticker,file_name,call_datetime_header_raw,call_datetime_gmt_ts,call_datetime_et_ts,gmt_hour_minute,after_market_close_et,event_trading_day_final,expected_event_trading_day,event_day_matches
77,CSCO,2016-May-18-CSCO.txt,"MAY 18, 2016 / 8:30PM GMT",2016-05-18 20:30:00+00:00,2016-05-18 16:30:00-04:00,20:30,True,2016-05-19,2016-05-19,True
78,CSCO,2016-Aug-17-CSCO.txt,"AUGUST 17, 2016 / 8:30PM GMT",2016-08-17 20:30:00+00:00,2016-08-17 16:30:00-04:00,20:30,True,2016-08-18,2016-08-18,True
81,CSCO,2017-May-17-CSCO.txt,"MAY 17, 2017 / 8:30PM GMT",2017-05-17 20:30:00+00:00,2017-05-17 16:30:00-04:00,20:30,True,2017-05-18,2017-05-18,True
82,CSCO,2017-Aug-16-CSCO.txt,"AUGUST 16, 2017 / 8:30PM GMT",2017-08-16 20:30:00+00:00,2017-08-16 16:30:00-04:00,20:30,True,2017-08-17,2017-08-17,True
85,CSCO,2018-May-16-CSCO.txt,"MAY 16, 2018 / 8:30PM GMT",2018-05-16 20:30:00+00:00,2018-05-16 16:30:00-04:00,20:30,True,2018-05-17,2018-05-17,True
...,...,...,...,...,...,...,...,...,...,...
53,AMZN,2019-Oct-24-AMZN.txt,"OCTOBER 24, 2019 / 9:30PM GMT",2019-10-24 21:30:00+00:00,2019-10-24 17:30:00-04:00,21:30,True,2019-10-25,2019-10-25,True
55,AMZN,2020-Apr-30-AMZN.txt,"APRIL 30, 2020 / 9:30PM GMT",2020-04-30 21:30:00+00:00,2020-04-30 17:30:00-04:00,21:30,True,2020-05-01,2020-05-01,True
56,AMZN,2020-Jul-30-AMZN.txt,"JULY 30, 2020 / 9:30PM GMT",2020-07-30 21:30:00+00:00,2020-07-30 17:30:00-04:00,21:30,True,2020-07-31,2020-07-31,True
76,CSCO,2016-Feb-10-CSCO.txt,"FEBRUARY 10, 2016 / 9:30PM GMT",2016-02-10 21:30:00+00:00,2016-02-10 16:30:00-05:00,21:30,True,2016-02-11,2016-02-11,True


## 9. Summary diagnostics

In [21]:
summary = {
    "total_rows": len(df),
    "rows_with_no_errors": int(df["error"].isna().sum()),
    "rows_with_errors": int(df["error"].notna().sum()),
    "rows_with_valid_gmt_timestamp": int(df["call_datetime_gmt_ts"].notna().sum()),
    "after_market_flag_matches": int(df["after_market_close_matches"].sum()),
    "after_market_flag_mismatches": int((~df["after_market_close_matches"]).sum()),
    "event_day_matches": int(df["event_day_matches"].sum()),
    "event_day_mismatches": int((~df["event_day_matches"]).sum()),
    "business_day_event_dates": int(df["event_day_is_business_day"].sum()),
    "non_business_day_event_dates": int((~df["event_day_is_business_day"]).sum()),
}

summary

{'total_rows': 188,
 'rows_with_no_errors': 188,
 'rows_with_errors': 0,
 'rows_with_valid_gmt_timestamp': 188,
 'after_market_flag_matches': 188,
 'after_market_flag_mismatches': 0,
 'event_day_matches': 188,
 'event_day_mismatches': 0,
 'business_day_event_dates': 188,
 'non_business_day_event_dates': 0}

## Validation Results

The event metadata pipeline was validated by independently recomputing
the after-market indicator and event trading day from the stored
timestamps.

All validation checks passed:

- 188 / 188 transcripts were parsed successfully
- 188 / 188 timestamps were valid and timezone-convertible
- 188 / 188 after-market flags matched the recomputed values
- 188 / 188 event trading days matched the independently recomputed rule
- 188 / 188 event days fall on valid business days

These results confirm that the event alignment logic implemented in
`scripts/build_event_metadata.py` correctly converts transcript timestamps
from GMT to `America/New_York` and assigns the appropriate event trading
day relative to the 4:00 PM U.S. market close.

The file `data/processed/event_metadata_final.csv` is therefore used as
the canonical event-level metadata table in all downstream analysis.